In [15]:
import pandas as pd 
import numpy as np 

data=pd.read_csv("/home/loubna/Code_Projet_Mathia/Mathia/data/Rawson/Rawson_et_al_2018_dat_vF.csv")

In [16]:
sess1=data[data["Session_Number"]==1]
sess2=data[data["Session_Number"]==2]
sess3=data[data["Session_Number"]==3]
sess4=data[data["Session_Number"]==4]
sess5=data[data["Session_Number"]==5]
sess6=data[data["Session_Number"]==6]


In [17]:
sess1 = sess1.sort_values(
    ["user_id", "order_id"]
).copy()

sess1["RT"] = pd.to_numeric(
    sess1["RT"],
    errors="coerce"
)

sess1["time_seconds"] = (
    sess1.groupby("user_id")["RT"]
         .transform(lambda x: x.cumsum().shift(fill_value=0))
)
origin = pd.Timestamp("2019-01-01 09:00:00")

sess1["timestamp"] = (
    origin +
    pd.to_timedelta(sess1["time_seconds"], unit="s")
)

In [18]:
def last_sess(sess1, sess2,days_wait):
    last_s1 = (sess1.sort_values(["user_id", "order_id"]).groupby("user_id").tail(1)[["user_id", "timestamp", "RT"]].copy())
    last_s1["end_s1"] = (last_s1["timestamp"]+ pd.to_timedelta(last_s1["RT"], unit="s"))
    last_s1["start_s2"] = (last_s1["end_s1"] + pd.Timedelta(days=days_wait))
    sess2 = sess2.merge(last_s1[["user_id", "start_s2"]], on="user_id",how="left")
    sess2 = sess2.sort_values(["user_id", "order_id"]).copy()
    sess2["RT"] = pd.to_numeric(sess2["RT"],errors="coerce")
    sess2["RT"] = sess2["RT"].fillna(sess2["RT"].median())
    sess2["time_seconds"] = (sess2.groupby("user_id")["RT"].transform(lambda x: x.cumsum().shift(fill_value=0)))
    sess2["timestamp"] = (sess2["start_s2"]+ pd.to_timedelta(sess2["time_seconds"], unit="s"))
    return sess2


In [19]:
sess2=last_sess(sess1,sess2,days_wait=7)

In [ ]:
sess2

In [ ]:
sess3=last_sess(sess2,sess3,days_wait=7)
sess3

In [ ]:
sess4=last_sess(sess3,sess4,days_wait=7)
sess4

In [ ]:
sess5=last_sess(sess4,sess5,days_wait=7)
sess5

In [ ]:
sess6=last_sess(sess5,sess6,days_wait=21)
sess6

In [27]:

sess2_sess3 = pd.concat([ sess2,sess3], axis=0, ignore_index=True)
sess2_sess3_sess4 = pd.concat([sess2, sess3,sess4], axis=0, ignore_index=True)
sess2_sess3_sess4_sess5 = pd.concat([sess2, sess3,sess4,sess5], axis=0, ignore_index=True)


In [32]:
def rename_col(sess):
    sess=sess.rename(columns={"Stim_ID":"item_id","skill_name":"KC"})
    sess=sess[["user_id","item_id","KC","correct","Session_Number","Learning_Condition_Short_Label",'Trial_N',"order_id","RT","timestamp","Assigned_Criterion"]]
    return sess

In [33]:
sess2=rename_col(sess2)
sess3=rename_col(sess3)
sess4=rename_col(sess4)
sess5=rename_col(sess5)
sess6=rename_col(sess6)
sess2_sess3=rename_col(sess2_sess3)
sess2_sess3_sess4=rename_col(sess2_sess3_sess4)
sess2_sess3_sess4_sess5=rename_col(sess2_sess3_sess4_sess5)


In [34]:
from pathlib import Path
source_file = Path("/home/loubna/Code_Projet_Mathia/Mathia/data/Rawson/Rawson_et_al_2018_dat_vF.csv")
out_dir = source_file.parent
frames = {
    "sess1": sess1,
    "sess2": sess2,
    "sess3": sess3,
    "sess4": sess4,
    "sess5": sess5,
    "sess6": sess6,
    "sess2_sess3": sess2_sess3,
    "sess2_sess3_sess4": sess2_sess3_sess4,
    "sess2_sess3_sess4_sess5": sess2_sess3_sess4_sess5,
}
for name, df in frames.items():
    df.to_csv(out_dir / f"{name}.csv", index=False)
all_sessions = pd.concat(
    [sess2, sess3, sess4, sess5, sess6],
    axis=0,
    ignore_index=True
)
all_sessions.to_csv(out_dir / "all_sessions.csv", index=False)